# Notebook 04: Interconnector Flow Analysis

## Project
NEM Network Constraints & Price Divergence Intelligence System

## Business Question
How do interconnector flows influence regional price behaviour, price spikes, and potential congestion signals between NSW1 and VIC1?

## Purpose Of This Notebook
Notebook 04 analyses interconnector results from `raw.interconnector_results`.

In Notebook 01, we built a clean base market table for NSW1 and VIC1.

In Notebook 02, we created base market features such as price spikes, supply-demand gap, price change, and rolling volatility.

In Notebook 03, we added constraint activity features to the market dataset.

This notebook adds the next network layer: interconnector flow behaviour.

The goal is to understand whether price events occur during periods of high flow, rapid flow changes, or possible congestion conditions.


## How This Notebook Links To The Full Project

The full project is designed to explain whether price outcomes are driven by normal regional market fundamentals or by network conditions.

Interconnectors are important because they allow electricity to flow between NEM regions. When interconnector capability is limited, regional prices can separate. A high-price region may not be able to import enough lower-cost energy from neighbouring regions.

This notebook supports three later project tasks:

1. Regional price divergence analysis.
2. Spike classification.
3. Decision intelligence recommendations.

The output from this notebook will later be joined with NSW1 and VIC1 price data to test whether price spikes or price separation occurred during stressed interconnector conditions.


## Key Market Concepts

### Interconnector
An interconnector is a transmission link between NEM regions. It allows power to flow from one region to another.

### Flow
Interconnector flow measures the MW transfer across the interconnector. Positive or negative values indicate direction depending on the interconnector convention.

### Flow change
Flow change measures the movement in interconnector flow from one dispatch interval to the next. Large changes can indicate rapidly changing market or network conditions.

### Import and export exposure
If a region is relying on imports, then reduced interconnector capability can increase price risk.

### Congestion signal
In this notebook, congestion is treated as an analytical signal rather than a formal AEMO constraint classification. We will look for high absolute flow and large flow changes as early indicators.


## Analyst Objective

By the end of this notebook, we want to create interconnector features that describe the level, direction, movement, and stress of interconnector flows.

These features will help answer:

- Were high-price intervals associated with high interconnector flow?
- Did flows change sharply during price events?
- Were there congestion-style signals during NSW1 price spikes?
- Which interconnectors are most relevant for NSW1 and VIC1 analysis?

### Feature Definitions And Formulas

| Feature | Definition | Formula / Rule | Analyst Meaning |
|---|---|---|---|
| `flow` | The MW flow on an interconnector for a dispatch interval. | `flow = mwflow` | Shows the direction and size of transfer across the interconnector. Positive and negative direction depends on AEMO interconnector convention. |
| `absolute_flow` | The size of the interconnector flow regardless of direction. | `absolute_flow = abs(flow)` | Measures how heavily the interconnector is being used, independent of import/export direction. |
| `flow_change` | The change in flow from the previous dispatch interval for the same interconnector. | `flow_change = current flow - previous interval flow` | Identifies whether interconnector transfer is moving quickly between intervals. |
| `absolute_flow_change` | The size of the flow movement regardless of direction. | `absolute_flow_change = abs(flow_change)` | Flags sudden flow swings, which may indicate changing dispatch, constraints, or price separation conditions. |
| `high_flow_flag` | Indicates unusually high flow for that interconnector. | `absolute_flow >= 90th percentile absolute flow for the interconnector` | Suggests the interconnector is being used heavily relative to its own observed behaviour. |
| `high_flow_change_flag` | Indicates unusually large movement in interconnector flow. | `absolute_flow_change >= 90th percentile absolute flow change for the interconnector` | Suggests rapid change in transfer conditions between intervals. |
| `congestion_signal_flag` | Practical congestion-style signal based on high flow or sharp flow movement. | `high_flow_flag == True or high_flow_change_flag == True` | Indicates intervals where interconnector behaviour may be relevant to price outcomes or regional separation. |

### Important Analytical Note

These flags are analytical indicators, not formal AEMO constraint classifications.

A formal congestion assessment would require interconnector limits, constraint equations, and directional limit information. In this notebook, we use high observed flow and sharp flow changes as practical early warning signals for market analysis.


In [1]:
from pathlib import Path
import os

import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 100)


In [2]:
PROJECT_ROOT = (
    Path.cwd().parents[0]
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"

MARKET_WITH_CONSTRAINTS_FILE = OUTPUT_DIR / "03_market_features_with_constraints.csv"

START_DATE = "2026-02-01"
END_DATE = "2026-03-01"
REGIONS = ["NSW1", "VIC1"]

print("Project root:", PROJECT_ROOT)
print("Input file:", MARKET_WITH_CONSTRAINTS_FILE)
print("Output directory:", OUTPUT_DIR)
print("Date range:", START_DATE, "to", END_DATE)
print("Regions:", REGIONS)


Project root: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence
Input file: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/03_market_features_with_constraints.csv
Output directory: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs
Date range: 2026-02-01 to 2026-03-01
Regions: ['NSW1', 'VIC1']


In [4]:
load_dotenv(PROJECT_ROOT / ".env")

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Database engine created.")


Database engine created.


In [5]:
with engine.connect() as conn:
    result = conn.execute(text("""
        select
            current_database() as database_name,
            current_user as user_name
    """))
    for row in result:
        print(row)


('nemweb', 'vivekarya')


## Step 5: Load Market Features With Constraint Features

### What we are doing
We are loading the output from Notebook 03.

### Why it matters
This table already contains price, demand, volatility, spike flags, and constraint features. Interconnector features will be added to this table later.


In [6]:
market_with_constraints = pd.read_csv(
    MARKET_WITH_CONSTRAINTS_FILE,
    parse_dates=["settlementdate"]
)

print("Market with constraints shape:", market_with_constraints.shape)

market_with_constraints.head()


Market with constraints shape: (16126, 21)


,settlementdate,regionid,rrp,intervention,totaldemand,availablegeneration,netinterchange,intervention_regionsum,supply_demand_gap,price_spike_flag,price_change,rolling_rrp_volatility_1h,date,hour,weekday,is_weekend,active_constraint_count,max_constraint_marginal_value,total_constraint_marginal_value,violation_flag,max_violation_degree
0,2026-02-01 00:05:00,NSW1,57.05984,0,8186.33,13272.53373,-1054.73,0,5086.20373,False,NaN,NaN,2026-02-01,0,Sunday,True,17,135.3015,-902.14975,False,0
1,2026-02-01 00:10:00,NSW1,64.51979,0,8165.29,13246.75420,-924.67,0,5081.46420,False,7.45995,NaN,2026-02-01,0,Sunday,True,16,40.2794,-1006.87256,False,0
2,2026-02-01 00:15:00,NSW1,65.08727,0,8202.70,13198.51267,-1006.50,0,4995.81267,False,0.56748,4.479816,2026-02-01,0,Sunday,True,20,9.1600,-1053.75439,False,0
3,2026-02-01 00:20:00,NSW1,64.89000,0,8213.41,13169.71135,-1011.73,0,4956.30135,False,-0.19727,3.893369,2026-02-01,0,Sunday,True,20,4.3600,-1061.42631,False,0
4,2026-02-01 00:25:00,NSW1,63.47180,0,8055.53,13139.84760,-934.04,0,5084.31760,False,-1.41820,3.381808,2026-02-01,0,Sunday,True,20,4.3600,-1060.65174,False,0


## Step 6: Inspect Interconnector Results Table Columns

### What we are doing
We are checking the columns available in `raw.interconnector_results`.

### Why it matters
Different ETL pipelines can store AEMO fields with slightly different names. Before extracting flow data, we need to confirm the column names for interconnector ID and MW flow.


In [7]:
interconnector_columns_query = text("""
select
    column_name,
    data_type
from information_schema.columns
where table_schema = 'raw'
  and table_name = 'interconnector_results'
order by ordinal_position;
""")

interconnector_columns_df = pd.read_sql(interconnector_columns_query, engine)

interconnector_columns_df


,column_name,data_type
0,settlementdate,timestamp without time zone
1,interconnectorid,text
2,mwflow,numeric
3,lastchanged,text
4,periodid,bigint
5,meteredmwflow,numeric
6,mwlosses,numeric


## Step 7: Check Interconnector Row Count

### What we are doing
We are checking how many interconnector records exist for the February 2026 study period.

### Why it matters
This confirms that the ETL successfully loaded interconnector results and that the selected date range has data.


In [8]:
interconnector_row_count_query = text("""
select
    count(*) as row_count
from raw.interconnector_results
where settlementdate >= :start_date
  and settlementdate < :end_date;
""")

interconnector_row_count_df = pd.read_sql(
    interconnector_row_count_query,
    engine,
    params={
        "start_date": START_DATE,
        "end_date": END_DATE,
    },
)

interconnector_row_count_df


,row_count
0,48348


## Step 8: Identify Available Interconnectors

### What we are doing
We are listing interconnector IDs available in the study period.

### Why it matters
NSW1 and VIC1 price behaviour may be influenced by several neighbouring interconnectors. Before filtering, we need to see which interconnector IDs exist in the data.


In [9]:
interconnector_id_query = text("""
select
    interconnectorid,
    count(*) as row_count,
    min(settlementdate) as first_interval,
    max(settlementdate) as last_interval
from raw.interconnector_results
where settlementdate >= :start_date
  and settlementdate < :end_date
group by interconnectorid
order by interconnectorid;
""")

interconnector_ids_df = pd.read_sql(
    interconnector_id_query,
    engine,
    params={
        "start_date": START_DATE,
        "end_date": END_DATE,
    },
)

interconnector_ids_df


,interconnectorid,row_count,first_interval,last_interval
0,N-Q-MNSP1,8058,2026-02-01 00:30:00,2026-02-28 23:55:00
1,NSW1-QLD1,8058,2026-02-01 00:30:00,2026-02-28 23:55:00
2,T-V-MNSP1,8058,2026-02-01 00:30:00,2026-02-28 23:55:00
3,V-S-MNSP1,8058,2026-02-01 00:30:00,2026-02-28 23:55:00
4,V-SA,8058,2026-02-01 00:30:00,2026-02-28 23:55:00
5,VIC1-NSW1,8058,2026-02-01 00:30:00,2026-02-28 23:55:00


## Step 9: Extract Interconnector Flow Data

### What we are doing
We are extracting dispatch interval interconnector flow records.

### Why it matters
This gives us the raw transfer behaviour between NEM regions. Later we will engineer features such as absolute flow, flow change, and congestion-style signals.


In [10]:
interconnector_query = text("""
select
    settlementdate,
    interconnectorid,
    mwflow
from raw.interconnector_results
where settlementdate >= :start_date
  and settlementdate < :end_date
order by interconnectorid, settlementdate;
""")

interconnector_df = pd.read_sql(
    interconnector_query,
    engine,
    params={
        "start_date": START_DATE,
        "end_date": END_DATE,
    },
)

print("Interconnector data shape:", interconnector_df.shape)

interconnector_df.head()


Interconnector data shape: (48348, 3)


,settlementdate,interconnectorid,mwflow
0,2026-02-01 00:30:00,N-Q-MNSP1,0.0
1,2026-02-01 00:35:00,N-Q-MNSP1,9.0
2,2026-02-01 00:40:00,N-Q-MNSP1,0.0
3,2026-02-01 00:45:00,N-Q-MNSP1,9.0
4,2026-02-01 00:50:00,N-Q-MNSP1,9.0


## Step 10: Clean Interconnector Flow Data

### What we are doing
We are standardising timestamps, interconnector IDs, and MW flow values.

### Why it matters
Clean and consistently ordered interconnector data is needed before calculating flow changes and high-flow indicators.


In [11]:
interconnector_clean = interconnector_df.copy()

interconnector_clean.columns = interconnector_clean.columns.str.lower()

interconnector_clean["settlementdate"] = pd.to_datetime(
    interconnector_clean["settlementdate"]
)

interconnector_clean["interconnectorid"] = (
    interconnector_clean["interconnectorid"]
    .astype(str)
    .str.upper()
    .str.strip()
)

interconnector_clean["mwflow"] = pd.to_numeric(
    interconnector_clean["mwflow"],
    errors="coerce"
)

interconnector_clean = interconnector_clean.dropna(
    subset=["settlementdate", "interconnectorid", "mwflow"]
)

interconnector_clean = interconnector_clean.drop_duplicates(
    subset=["settlementdate", "interconnectorid"]
)

interconnector_clean = interconnector_clean.sort_values(
    ["interconnectorid", "settlementdate"]
).reset_index(drop=True)

interconnector_clean.head()


,settlementdate,interconnectorid,mwflow
0,2026-02-01 00:30:00,N-Q-MNSP1,0.0
1,2026-02-01 00:35:00,N-Q-MNSP1,9.0
2,2026-02-01 00:40:00,N-Q-MNSP1,0.0
3,2026-02-01 00:45:00,N-Q-MNSP1,9.0
4,2026-02-01 00:50:00,N-Q-MNSP1,9.0


## Step 11: Rename MW Flow To Flow

### What we are doing
We are renaming `mwflow` to `flow`.

### Why it matters
The name `flow` is easier to use in feature engineering and reporting. It represents the MW transfer on the interconnector for each dispatch interval.


In [12]:
interconnector_features = interconnector_clean.rename(
    columns={"mwflow": "flow"}
)

interconnector_features.head()


,settlementdate,interconnectorid,flow
0,2026-02-01 00:30:00,N-Q-MNSP1,0.0
1,2026-02-01 00:35:00,N-Q-MNSP1,9.0
2,2026-02-01 00:40:00,N-Q-MNSP1,0.0
3,2026-02-01 00:45:00,N-Q-MNSP1,9.0
4,2026-02-01 00:50:00,N-Q-MNSP1,9.0


## Step 12: Create Absolute Flow

### What we are doing
We are calculating the absolute size of interconnector flow, ignoring direction.

### Why it matters
Interconnector flow can be positive or negative depending on direction. Absolute flow tells us how heavily the interconnector is being used regardless of import or export direction.

### Formula
absolute_flow = abs(flow)


In [13]:
interconnector_features["absolute_flow"] = (
    interconnector_features["flow"].abs()
)

interconnector_features[
    [
        "settlementdate",
        "interconnectorid",
        "flow",
        "absolute_flow",
    ]
].head()


,settlementdate,interconnectorid,flow,absolute_flow
0,2026-02-01 00:30:00,N-Q-MNSP1,0.0,0.0
1,2026-02-01 00:35:00,N-Q-MNSP1,9.0,9.0
2,2026-02-01 00:40:00,N-Q-MNSP1,0.0,0.0
3,2026-02-01 00:45:00,N-Q-MNSP1,9.0,9.0
4,2026-02-01 00:50:00,N-Q-MNSP1,9.0,9.0


## Step 13: Create Flow Change

### What we are doing
We are calculating the change in flow from the previous dispatch interval for each interconnector.

### Why it matters
Large flow changes can indicate rapidly changing dispatch conditions, interconnector response to price differences, or constraint-driven movement.

### Formula
flow_change = current interval flow - previous interval flow


In [14]:
interconnector_features["flow_change"] = (
    interconnector_features
    .groupby("interconnectorid")["flow"]
    .diff()
)

interconnector_features[
    [
        "settlementdate",
        "interconnectorid",
        "flow",
        "flow_change",
    ]
].head(10)


,settlementdate,interconnectorid,flow,flow_change
0,2026-02-01 00:30:00,N-Q-MNSP1,0.0,NaN
1,2026-02-01 00:35:00,N-Q-MNSP1,9.0,9.0
2,2026-02-01 00:40:00,N-Q-MNSP1,0.0,-9.0
3,2026-02-01 00:45:00,N-Q-MNSP1,9.0,9.0
4,2026-02-01 00:50:00,N-Q-MNSP1,9.0,0.0
5,2026-02-01 00:55:00,N-Q-MNSP1,0.0,-9.0
6,2026-02-01 01:00:00,N-Q-MNSP1,9.0,9.0
7,2026-02-01 01:05:00,N-Q-MNSP1,0.0,-9.0
8,2026-02-01 01:10:00,N-Q-MNSP1,0.0,0.0
9,2026-02-01 01:15:00,N-Q-MNSP1,0.0,0.0


## Step 14: Create Absolute Flow Change

### What we are doing
We are calculating the size of flow movement regardless of direction.

### Why it matters
A large positive or negative change can both indicate material movement in interconnector behaviour. Absolute flow change helps us identify large swings without focusing on direction.

### Formula
absolute_flow_change = abs(flow_change)


In [15]:
interconnector_features["absolute_flow_change"] = (
    interconnector_features["flow_change"].abs()
)

interconnector_features[
    [
        "settlementdate",
        "interconnectorid",
        "flow",
        "flow_change",
        "absolute_flow_change",
    ]
].head(10)


,settlementdate,interconnectorid,flow,flow_change,absolute_flow_change
0,2026-02-01 00:30:00,N-Q-MNSP1,0.0,NaN,NaN
1,2026-02-01 00:35:00,N-Q-MNSP1,9.0,9.0,9.0
2,2026-02-01 00:40:00,N-Q-MNSP1,0.0,-9.0,9.0
3,2026-02-01 00:45:00,N-Q-MNSP1,9.0,9.0,9.0
4,2026-02-01 00:50:00,N-Q-MNSP1,9.0,0.0,0.0
5,2026-02-01 00:55:00,N-Q-MNSP1,0.0,-9.0,9.0
6,2026-02-01 01:00:00,N-Q-MNSP1,9.0,9.0,9.0
7,2026-02-01 01:05:00,N-Q-MNSP1,0.0,-9.0,9.0
8,2026-02-01 01:10:00,N-Q-MNSP1,0.0,0.0,0.0
9,2026-02-01 01:15:00,N-Q-MNSP1,0.0,0.0,0.0


## Step 15: Create High Flow Thresholds

### What we are doing
We are calculating the 90th percentile absolute flow for each interconnector.

### Why it matters
Different interconnectors have different normal flow ranges. A single fixed MW threshold would be misleading. Using each interconnector's own 90th percentile identifies periods when that interconnector is heavily used relative to its own observed behaviour.


In [16]:
flow_thresholds = (
    interconnector_features
    .groupby("interconnectorid")
    .agg(
        high_flow_threshold=("absolute_flow", lambda x: x.quantile(0.90)),
        high_flow_change_threshold=("absolute_flow_change", lambda x: x.quantile(0.90)),
    )
    .reset_index()
)

flow_thresholds


,interconnectorid,high_flow_threshold,high_flow_change_threshold
0,N-Q-MNSP1,65.000,16.000
1,NSW1-QLD1,787.090,149.416
2,T-V-MNSP1,195.906,42.218
3,V-S-MNSP1,175.229,45.374
4,V-SA,554.002,98.974
5,VIC1-NSW1,1069.641,165.084


## Step 16: Add High Flow Thresholds To Interconnector Features

### What we are doing
We are joining each interconnector's high-flow and high-flow-change thresholds back onto the interval-level flow table.

### Why it matters
This lets each interval be compared against the relevant threshold for that specific interconnector.


In [17]:
interconnector_features = interconnector_features.merge(
    flow_thresholds,
    on="interconnectorid",
    how="left"
)

interconnector_features.head()


,settlementdate,interconnectorid,flow,absolute_flow,flow_change,absolute_flow_change,high_flow_threshold,high_flow_change_threshold
0,2026-02-01 00:30:00,N-Q-MNSP1,0.0,0.0,NaN,NaN,65.0,16.0
1,2026-02-01 00:35:00,N-Q-MNSP1,9.0,9.0,9.0,9.0,65.0,16.0
2,2026-02-01 00:40:00,N-Q-MNSP1,0.0,0.0,-9.0,9.0,65.0,16.0
3,2026-02-01 00:45:00,N-Q-MNSP1,9.0,9.0,9.0,9.0,65.0,16.0
4,2026-02-01 00:50:00,N-Q-MNSP1,9.0,9.0,0.0,0.0,65.0,16.0


## Step 17: Create High Flow Flag

### What we are doing
We are flagging intervals where absolute flow is greater than or equal to the interconnector's 90th percentile flow threshold.

### Why it matters
This identifies intervals when an interconnector is being heavily used relative to its own normal February 2026 behaviour.

### Rule
high_flow_flag = absolute_flow >= high_flow_threshold


In [18]:
interconnector_features["high_flow_flag"] = (
    interconnector_features["absolute_flow"]
    >= interconnector_features["high_flow_threshold"]
)

interconnector_features[
    [
        "settlementdate",
        "interconnectorid",
        "flow",
        "absolute_flow",
        "high_flow_threshold",
        "high_flow_flag",
    ]
].head()


,settlementdate,interconnectorid,flow,absolute_flow,high_flow_threshold,high_flow_flag
0,2026-02-01 00:30:00,N-Q-MNSP1,0.0,0.0,65.0,False
1,2026-02-01 00:35:00,N-Q-MNSP1,9.0,9.0,65.0,False
2,2026-02-01 00:40:00,N-Q-MNSP1,0.0,0.0,65.0,False
3,2026-02-01 00:45:00,N-Q-MNSP1,9.0,9.0,65.0,False
4,2026-02-01 00:50:00,N-Q-MNSP1,9.0,9.0,65.0,False


## Step 18: Create High Flow Change Flag

### What we are doing
We are flagging intervals where absolute flow change is greater than or equal to the interconnector's 90th percentile flow-change threshold.

### Why it matters
This identifies intervals with unusually sharp changes in interconnector transfer. Sharp flow movements can occur around changing regional prices, constraints, or dispatch conditions.

### Rule
high_flow_change_flag = absolute_flow_change >= high_flow_change_threshold


In [19]:
interconnector_features["high_flow_change_flag"] = (
    interconnector_features["absolute_flow_change"]
    >= interconnector_features["high_flow_change_threshold"]
)

interconnector_features[
    [
        "settlementdate",
        "interconnectorid",
        "flow",
        "flow_change",
        "absolute_flow_change",
        "high_flow_change_threshold",
        "high_flow_change_flag",
    ]
].head(10)


,settlementdate,interconnectorid,flow,flow_change,absolute_flow_change,high_flow_change_threshold,high_flow_change_flag
0,2026-02-01 00:30:00,N-Q-MNSP1,0.0,NaN,NaN,16.0,False
1,2026-02-01 00:35:00,N-Q-MNSP1,9.0,9.0,9.0,16.0,False
2,2026-02-01 00:40:00,N-Q-MNSP1,0.0,-9.0,9.0,16.0,False
3,2026-02-01 00:45:00,N-Q-MNSP1,9.0,9.0,9.0,16.0,False
4,2026-02-01 00:50:00,N-Q-MNSP1,9.0,0.0,0.0,16.0,False
5,2026-02-01 00:55:00,N-Q-MNSP1,0.0,-9.0,9.0,16.0,False
6,2026-02-01 01:00:00,N-Q-MNSP1,9.0,9.0,9.0,16.0,False
7,2026-02-01 01:05:00,N-Q-MNSP1,0.0,-9.0,9.0,16.0,False
8,2026-02-01 01:10:00,N-Q-MNSP1,0.0,0.0,0.0,16.0,False
9,2026-02-01 01:15:00,N-Q-MNSP1,0.0,0.0,0.0,16.0,False


## Step 19: Create Congestion Signal Flag

### What we are doing
We are creating a practical congestion-style signal based on high absolute flow or unusually large flow movement.

### Why it matters
Without formal interconnector limit data, this flag is not a formal congestion classification. It is an analytical signal that interconnector behaviour may be relevant to regional price outcomes.

### Rule
congestion_signal_flag = high_flow_flag OR high_flow_change_flag


In [20]:
interconnector_features["congestion_signal_flag"] = (
    interconnector_features["high_flow_flag"]
    | interconnector_features["high_flow_change_flag"]
)

interconnector_features[
    [
        "settlementdate",
        "interconnectorid",
        "flow",
        "absolute_flow",
        "flow_change",
        "absolute_flow_change",
        "high_flow_flag",
        "high_flow_change_flag",
        "congestion_signal_flag",
    ]
].head(10)


,settlementdate,interconnectorid,flow,absolute_flow,flow_change,absolute_flow_change,high_flow_flag,high_flow_change_flag,congestion_signal_flag
0,2026-02-01 00:30:00,N-Q-MNSP1,0.0,0.0,NaN,NaN,False,False,False
1,2026-02-01 00:35:00,N-Q-MNSP1,9.0,9.0,9.0,9.0,False,False,False
2,2026-02-01 00:40:00,N-Q-MNSP1,0.0,0.0,-9.0,9.0,False,False,False
3,2026-02-01 00:45:00,N-Q-MNSP1,9.0,9.0,9.0,9.0,False,False,False
4,2026-02-01 00:50:00,N-Q-MNSP1,9.0,9.0,0.0,0.0,False,False,False
5,2026-02-01 00:55:00,N-Q-MNSP1,0.0,0.0,-9.0,9.0,False,False,False
6,2026-02-01 01:00:00,N-Q-MNSP1,9.0,9.0,9.0,9.0,False,False,False
7,2026-02-01 01:05:00,N-Q-MNSP1,0.0,0.0,-9.0,9.0,False,False,False
8,2026-02-01 01:10:00,N-Q-MNSP1,0.0,0.0,0.0,0.0,False,False,False
9,2026-02-01 01:15:00,N-Q-MNSP1,0.0,0.0,0.0,0.0,False,False,False


## Step 20: Summarise Interconnector Flow Behaviour

### What we are doing
We are creating a summary table by interconnector.

### Why it matters
This shows which interconnectors had the largest flows, largest flow changes, and most frequent congestion-style signals during the study period.


In [21]:
interconnector_flow_summary = (
    interconnector_features
    .groupby("interconnectorid")
    .agg(
        intervals=("settlementdate", "count"),
        average_flow=("flow", "mean"),
        min_flow=("flow", "min"),
        max_flow=("flow", "max"),
        average_absolute_flow=("absolute_flow", "mean"),
        max_absolute_flow=("absolute_flow", "max"),
        average_absolute_flow_change=("absolute_flow_change", "mean"),
        max_absolute_flow_change=("absolute_flow_change", "max"),
        high_flow_intervals=("high_flow_flag", "sum"),
        high_flow_change_intervals=("high_flow_change_flag", "sum"),
        congestion_signal_intervals=("congestion_signal_flag", "sum"),
    )
    .reset_index()
)

interconnector_flow_summary["congestion_signal_share_pct"] = (
    interconnector_flow_summary["congestion_signal_intervals"]
    / interconnector_flow_summary["intervals"]
    * 100
)

interconnector_flow_summary


,interconnectorid,intervals,average_flow,min_flow,max_flow,average_absolute_flow,max_absolute_flow,average_absolute_flow_change,max_absolute_flow_change,high_flow_intervals,high_flow_change_intervals,congestion_signal_intervals,congestion_signal_share_pct
0,N-Q-MNSP1,8058,2.364829,-201.60,96.98,30.458304,201.60,6.561460,117.70,913,840,1557,19.322413
1,NSW1-QLD1,8058,-152.740848,-1298.52,950.00,372.618797,1298.52,64.687101,674.01,806,806,1555,19.297592
2,T-V-MNSP1,8058,-52.848345,-443.35,350.00,63.844415,443.35,11.655682,220.99,806,806,1446,17.944899
3,V-S-MNSP1,8058,72.406669,-159.79,213.25,103.896466,213.25,12.540732,117.00,806,806,1572,19.508563
4,V-SA,8058,156.755278,-683.32,698.41,314.067475,698.41,40.338578,686.14,806,806,1577,19.570613
5,VIC1-NSW1,8058,311.242658,-1872.63,1300.76,513.671018,1872.63,70.504534,1447.09,806,806,1598,19.831224


## Interpretation: Interconnector Flow Summary

The interconnector summary shows that several interconnectors experienced high-flow or high-flow-change conditions during the February 2026 study period.

For the NSW1 and VIC1 focus, the most important interconnector is `VIC1-NSW1`. This interconnector recorded the largest maximum absolute flow, reaching approximately 1,873 MW, and the largest absolute flow change, reaching approximately 1,447 MW between dispatch intervals. This indicates that the NSW-VIC transfer path experienced large directional and magnitude changes during the study period.

The `VIC1-NSW1` interconnector also had a congestion-style signal in around 19.8% of observed intervals using the analytical rule defined in this notebook. This does not prove formal congestion, because formal congestion would require directional limits and constraint equation analysis. However, it does indicate that interconnector behaviour is likely to be relevant when explaining NSW1-VIC1 price separation and NSW1 spike events.

Other interconnectors such as `NSW1-QLD1` and `V-SA` also showed material flow and flow-change behaviour, but for this project the `VIC1-NSW1` path is the primary focus because it directly connects the two regions being compared.

The next step is to aggregate interconnector signals to dispatch interval level and join them to the market and constraint feature table.


## Step 21: Aggregate Interconnector Signals To Dispatch Interval Level

### What we are doing
We are converting interconnector-level records into one network-flow feature row per dispatch interval.

### Why it matters
The market feature table has one row per region per dispatch interval. To join interconnector signals to market prices, we need interval-level interconnector features.


In [22]:
interconnector_interval_features = (
    interconnector_features
    .groupby("settlementdate")
    .agg(
        total_absolute_flow=("absolute_flow", "sum"),
        max_absolute_flow=("absolute_flow", "max"),
        average_absolute_flow=("absolute_flow", "mean"),
        max_absolute_flow_change=("absolute_flow_change", "max"),
        high_flow_interconnector_count=("high_flow_flag", "sum"),
        high_flow_change_interconnector_count=("high_flow_change_flag", "sum"),
        congestion_signal_count=("congestion_signal_flag", "sum"),
        congestion_signal_flag=("congestion_signal_flag", "any"),
    )
    .reset_index()
)

interconnector_interval_features.head()


,settlementdate,total_absolute_flow,max_absolute_flow,average_absolute_flow,max_absolute_flow_change,high_flow_interconnector_count,high_flow_change_interconnector_count,congestion_signal_count,congestion_signal_flag
0,2026-02-01 00:30:00,1241.36,971.18,206.893333,NaN,0,0,0,False
1,2026-02-01 00:35:00,1409.72,999.57,234.953333,92.97,0,2,2,True
2,2026-02-01 00:40:00,1363.19,1076.22,227.198333,110.32,1,2,3,True
3,2026-02-01 00:45:00,1470.92,1053.62,245.153333,87.39,0,1,1,True
4,2026-02-01 00:50:00,1393.82,1029.14,232.303333,29.13,0,0,0,False


## Step 22: Create VIC1-NSW1 Specific Features

### What we are doing
We are creating a focused feature table for the `VIC1-NSW1` interconnector.

### Why it matters
The project focuses on NSW1 and VIC1 price divergence. The `VIC1-NSW1` interconnector directly links those regions, so its flow behaviour is especially important for later price divergence and spike explanation.


In [23]:
vic_nsw_flow_features = (
    interconnector_features[
        interconnector_features["interconnectorid"] == "VIC1-NSW1"
    ]
    .copy()
    .rename(
        columns={
            "flow": "vic_nsw_flow",
            "absolute_flow": "vic_nsw_absolute_flow",
            "flow_change": "vic_nsw_flow_change",
            "absolute_flow_change": "vic_nsw_absolute_flow_change",
            "high_flow_flag": "vic_nsw_high_flow_flag",
            "high_flow_change_flag": "vic_nsw_high_flow_change_flag",
            "congestion_signal_flag": "vic_nsw_congestion_signal_flag",
        }
    )
)

vic_nsw_flow_features = vic_nsw_flow_features[
    [
        "settlementdate",
        "vic_nsw_flow",
        "vic_nsw_absolute_flow",
        "vic_nsw_flow_change",
        "vic_nsw_absolute_flow_change",
        "vic_nsw_high_flow_flag",
        "vic_nsw_high_flow_change_flag",
        "vic_nsw_congestion_signal_flag",
    ]
]

vic_nsw_flow_features.head()


,settlementdate,vic_nsw_flow,vic_nsw_absolute_flow,vic_nsw_flow_change,vic_nsw_absolute_flow_change,vic_nsw_high_flow_flag,vic_nsw_high_flow_change_flag,vic_nsw_congestion_signal_flag
40290,2026-02-01 00:30:00,971.18,971.18,NaN,NaN,False,False,False
40291,2026-02-01 00:35:00,999.57,999.57,28.39,28.39,False,False,False
40292,2026-02-01 00:40:00,1076.22,1076.22,76.65,76.65,True,False,True
40293,2026-02-01 00:45:00,1053.62,1053.62,-22.60,22.60,False,False,False
40294,2026-02-01 00:50:00,1029.14,1029.14,-24.48,24.48,False,False,False


## Step 23: Join Interconnector Features To Market And Constraint Features

### What we are doing
We are joining interval-level interconnector features and VIC1-NSW1 specific features onto the market and constraint dataset.

### Why it matters
This creates a single network-aware market table containing price, demand, volatility, constraint activity, and interconnector flow signals.


In [24]:
market_with_network = market_with_constraints.merge(
    interconnector_interval_features,
    on="settlementdate",
    how="left"
)

market_with_network = market_with_network.merge(
    vic_nsw_flow_features,
    on="settlementdate",
    how="left"
)

network_feature_cols = [
    "total_absolute_flow",
    "max_absolute_flow",
    "average_absolute_flow",
    "max_absolute_flow_change",
    "high_flow_interconnector_count",
    "high_flow_change_interconnector_count",
    "congestion_signal_count",
    "congestion_signal_flag",
    "vic_nsw_flow",
    "vic_nsw_absolute_flow",
    "vic_nsw_flow_change",
    "vic_nsw_absolute_flow_change",
    "vic_nsw_high_flow_flag",
    "vic_nsw_high_flow_change_flag",
    "vic_nsw_congestion_signal_flag",
]

market_with_network[network_feature_cols] = market_with_network[
    network_feature_cols
].fillna({
    "total_absolute_flow": 0,
    "max_absolute_flow": 0,
    "average_absolute_flow": 0,
    "max_absolute_flow_change": 0,
    "high_flow_interconnector_count": 0,
    "high_flow_change_interconnector_count": 0,
    "congestion_signal_count": 0,
    "congestion_signal_flag": False,
    "vic_nsw_flow": 0,
    "vic_nsw_absolute_flow": 0,
    "vic_nsw_flow_change": 0,
    "vic_nsw_absolute_flow_change": 0,
    "vic_nsw_high_flow_flag": False,
    "vic_nsw_high_flow_change_flag": False,
    "vic_nsw_congestion_signal_flag": False,
})

print("Market with constraints:", market_with_constraints.shape)
print("Market with network:", market_with_network.shape)

market_with_network.head()


Market with constraints: (16126, 21)
Market with network: (16126, 36)


/var/folders/m8/cwx1pw8d7qs9k5mbrk0vzxz80000gn/T/ipykernel_1682/1705187337.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ].fillna({


,settlementdate,regionid,rrp,intervention,totaldemand,availablegeneration,netinterchange,intervention_regionsum,supply_demand_gap,price_spike_flag,price_change,rolling_rrp_volatility_1h,date,hour,weekday,is_weekend,active_constraint_count,max_constraint_marginal_value,total_constraint_marginal_value,violation_flag,max_violation_degree,total_absolute_flow,max_absolute_flow,average_absolute_flow,max_absolute_flow_change,high_flow_interconnector_count,high_flow_change_interconnector_count,congestion_signal_count,congestion_signal_flag,vic_nsw_flow,vic_nsw_absolute_flow,vic_nsw_flow_change,vic_nsw_absolute_flow_change,vic_nsw_high_flow_flag,vic_nsw_high_flow_change_flag,vic_nsw_congestion_signal_flag
0,2026-02-01 00:05:00,NSW1,57.05984,0,8186.33,13272.53373,-1054.73,0,5086.20373,False,NaN,NaN,2026-02-01,0,Sunday,True,17,135.3015,-902.14975,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False
1,2026-02-01 00:10:00,NSW1,64.51979,0,8165.29,13246.75420,-924.67,0,5081.46420,False,7.45995,NaN,2026-02-01,0,Sunday,True,16,40.2794,-1006.87256,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False
2,2026-02-01 00:15:00,NSW1,65.08727,0,8202.70,13198.51267,-1006.50,0,4995.81267,False,0.56748,4.479816,2026-02-01,0,Sunday,True,20,9.1600,-1053.75439,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False
3,2026-02-01 00:20:00,NSW1,64.89000,0,8213.41,13169.71135,-1011.73,0,4956.30135,False,-0.19727,3.893369,2026-02-01,0,Sunday,True,20,4.3600,-1061.42631,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False
4,2026-02-01 00:25:00,NSW1,63.47180,0,8055.53,13139.84760,-934.04,0,5084.31760,False,-1.41820,3.381808,2026-02-01,0,Sunday,True,20,4.3600,-1060.65174,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False


## Step 24: Compare Interconnector Signals During Spike And Non-Spike Intervals

### What we are doing
We are comparing interconnector behaviour during price spike and non-spike intervals.

### Why it matters
If spike intervals show higher flows, sharper flow changes, or more congestion-style signals, interconnector behaviour may help explain price outcomes.


In [25]:
interconnector_vs_spike_summary = (
    market_with_network
    .groupby(["regionid", "price_spike_flag"])
    .agg(
        intervals=("settlementdate", "count"),
        average_rrp=("rrp", "mean"),
        max_rrp=("rrp", "max"),
        average_total_absolute_flow=("total_absolute_flow", "mean"),
        average_max_absolute_flow=("max_absolute_flow", "mean"),
        average_max_absolute_flow_change=("max_absolute_flow_change", "mean"),
        average_congestion_signal_count=("congestion_signal_count", "mean"),
        congestion_signal_intervals=("congestion_signal_flag", "sum"),
        average_vic_nsw_absolute_flow=("vic_nsw_absolute_flow", "mean"),
        average_vic_nsw_absolute_flow_change=("vic_nsw_absolute_flow_change", "mean"),
        vic_nsw_congestion_signal_intervals=("vic_nsw_congestion_signal_flag", "sum"),
    )
    .reset_index()
)

interconnector_vs_spike_summary


,regionid,price_spike_flag,intervals,average_rrp,max_rrp,average_total_absolute_flow,average_max_absolute_flow,average_max_absolute_flow_change,average_congestion_signal_count,congestion_signal_intervals,average_vic_nsw_absolute_flow,average_vic_nsw_absolute_flow_change,vic_nsw_congestion_signal_intervals
0,NSW1,False,8010,68.323133,299.82000,1397.727116,693.807514,107.522207,1.148814,5329,515.035758,70.481523,1587
1,NSW1,True,53,2417.098468,20300.00000,1391.959811,877.565849,142.607925,1.943396,49,258.955472,66.000566,11
2,VIC1,False,8063,45.055455,269.77601,1397.689206,695.015401,107.752834,1.154037,5378,513.352482,70.452069,1598


## Interpretation: Interconnector Signals During Spike And Non-Spike Intervals

The interconnector comparison shows that NSW1 recorded 53 price spike intervals above $300/MWh, while VIC1 recorded no spike intervals under the selected threshold during the February 2026 study period.

During NSW1 non-spike intervals, average RRP was about $68/MWh. During NSW1 spike intervals, average RRP increased sharply to about $2,417/MWh, with a maximum RRP of $20,300/MWh.

Interconnector behaviour changed during NSW1 spike intervals. The average maximum absolute interconnector flow increased from around 694 MW during NSW1 non-spike intervals to around 878 MW during NSW1 spike intervals. Average maximum absolute flow change also increased from around 108 MW to around 143 MW. This suggests that NSW1 spike intervals occurred during periods of stronger interconnector stress or sharper flow movement across the broader network.

The average congestion signal count also increased from around 1.15 during NSW1 non-spike intervals to around 1.94 during NSW1 spike intervals. In addition, 49 out of 53 NSW1 spike intervals had at least one congestion-style signal. This is a strong indication that interconnector behaviour was relevant during most NSW1 spike events.

However, the direct `VIC1-NSW1` interconnector tells a more nuanced story. Average `VIC1-NSW1` absolute flow was lower during NSW1 spike intervals than during non-spike intervals, falling from around 515 MW to around 259 MW. Average `VIC1-NSW1` flow change was also slightly lower during spike intervals. Only 11 of the 53 NSW1 spike intervals had a `VIC1-NSW1` congestion-style signal.

This suggests that NSW1 price spikes were associated with broader interconnector stress across the NEM, but not necessarily direct stress on the `VIC1-NSW1` interconnector alone. The next step is to analyse NSW1-VIC1 price divergence directly to see whether NSW1 prices separated from VIC1 during these spike intervals and whether that separation aligns with interconnector signals.


In [26]:
interconnector_features_output = OUTPUT_DIR / "04_interconnector_features.csv"
interconnector_summary_output = OUTPUT_DIR / "04_interconnector_flow_summary.csv"
interconnector_interval_output = OUTPUT_DIR / "04_interconnector_interval_features.csv"
interconnector_spike_summary_output = OUTPUT_DIR / "04_interconnector_vs_spike_summary.csv"
market_with_network_output = OUTPUT_DIR / "04_market_features_with_network.csv"

interconnector_features.to_csv(interconnector_features_output, index=False)
interconnector_flow_summary.to_csv(interconnector_summary_output, index=False)
interconnector_interval_features.to_csv(interconnector_interval_output, index=False)
interconnector_vs_spike_summary.to_csv(interconnector_spike_summary_output, index=False)
market_with_network.to_csv(market_with_network_output, index=False)

print("Saved:", interconnector_features_output)
print("Saved:", interconnector_summary_output)
print("Saved:", interconnector_interval_output)
print("Saved:", interconnector_spike_summary_output)
print("Saved:", market_with_network_output)


Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/04_interconnector_features.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/04_interconnector_flow_summary.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/04_interconnector_interval_features.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/04_interconnector_vs_spike_summary.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/04_market_features_with_network.csv


## Notebook 04 Summary: Interconnector Flow Analysis

This notebook added the interconnector flow layer to the Project 2 analysis. The purpose was to understand whether interconnector behaviour may help explain NSW1 price spikes, congestion-style conditions, and future NSW1-VIC1 price divergence.

| Step | What We Did | Why It Matters | Output / Feature Created |
|---|---|---|---|
| Step 1-5 | Loaded libraries, project settings, database connection, and the market dataset from Notebook 03. | This ensured the interconnector analysis used the same February 2026 study period and joined onto the existing market plus constraint feature table. | `market_with_constraints` |
| Step 6 | Inspected columns in `raw.interconnector_results`. | Confirmed the correct fields for interconnector ID and MW flow before extracting data. | Column validation |
| Step 7 | Checked interconnector row count for the study period. | Confirmed the ETL had loaded interconnector records for February 2026. | Row count validation |
| Step 8 | Listed available interconnectors. | Identified which interconnectors were present and confirmed `VIC1-NSW1` was available for NSW1-VIC1 analysis. | Interconnector ID list |
| Step 9 | Extracted `settlementdate`, `interconnectorid`, and `mwflow`. | Pulled the raw flow data needed to analyse transfer behaviour between NEM regions. | `interconnector_df` |
| Step 10 | Cleaned interconnector data. | Standardised timestamps, IDs, numeric flow values, duplicates, and sorting before feature engineering. | `interconnector_clean` |
| Step 11 | Renamed `mwflow` to `flow`. | Made the feature name easier to understand and use in later analysis. | `flow` |
| Step 12 | Created absolute flow. | Measured interconnector loading regardless of flow direction. | `absolute_flow = abs(flow)` |
| Step 13 | Created flow change. | Measured how much flow changed from the previous dispatch interval. | `flow_change` |
| Step 14 | Created absolute flow change. | Captured the size of flow movement regardless of direction. | `absolute_flow_change = abs(flow_change)` |
| Step 15 | Calculated 90th percentile flow thresholds by interconnector. | Created interconnector-specific benchmarks instead of using one fixed MW threshold for all links. | `high_flow_threshold`, `high_flow_change_threshold` |
| Step 16 | Joined thresholds back to interval-level flow data. | Allowed each dispatch interval to be compared against the correct threshold for its interconnector. | Threshold-enhanced flow table |
| Step 17 | Created high flow flag. | Flagged intervals where an interconnector was heavily loaded relative to its own observed behaviour. | `high_flow_flag` |
| Step 18 | Created high flow change flag. | Flagged intervals with unusually sharp flow movements. | `high_flow_change_flag` |
| Step 19 | Created congestion-style signal flag. | Identified intervals where interconnector behaviour may be relevant to price outcomes. | `congestion_signal_flag` |
| Step 20 | Summarised interconnector flow behaviour. | Showed which interconnectors had the largest flows, biggest flow swings, and most frequent congestion-style signals. | `interconnector_flow_summary` |
| Step 21 | Aggregated interconnector signals to dispatch interval level. | Converted many interconnector rows into one interval-level feature row for joining to market data. | `interconnector_interval_features` |
| Step 22 | Created `VIC1-NSW1` specific features. | Focused on the direct interconnector between the two regions in this project. | `vic_nsw_flow`, `vic_nsw_absolute_flow`, `vic_nsw_flow_change`, `vic_nsw_congestion_signal_flag` |
| Step 23 | Joined interconnector features to market and constraint features. | Built a combined market-network dataset with price, demand, volatility, constraints, and interconnector signals. | `market_with_network` |
| Step 24 | Compared interconnector signals during spike and non-spike intervals. | Tested whether price spikes coincided with higher flow, sharper flow changes, or congestion-style behaviour. | `interconnector_vs_spike_summary` |
| Step 25 | Saved Notebook 04 outputs. | Created reusable CSV files for later divergence analysis, spike classification, visualisation, and Power BI. | `04_interconnector_features.csv`, `04_interconnector_flow_summary.csv`, `04_interconnector_interval_features.csv`, `04_interconnector_vs_spike_summary.csv`, `04_market_features_with_network.csv` |

### Key Analyst Takeaway

Notebook 04 shows that interconnector behaviour is a meaningful network layer for this project. The `VIC1-NSW1` interconnector is especially important because it directly links the two focus regions and recorded large absolute flows and large flow changes during the study period.

The congestion-style flags created here are not formal AEMO congestion classifications. They are practical analytical signals based on high observed flow and sharp
